# W&B Run Plots

This notebook calls the reusable plotting helpers in `wandb_metrics.py`. The plot calls below correspond to the cells marked `#KEEP` in `../wandb_run_plots.ipynb`.


In [ ]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import wandb_metrics as wm
wm = importlib.reload(wm)
print("wandb_metrics:", wm.__file__)


## Choose Runs To Load Or Download


In [ ]:
import importlib
wm = importlib.reload(wm)

# Pick config folder(s) under Topology_Task/configs, or use None / "None" / "all" for every run.
# Examples:
# CONFIG_FOLDERS_TO_DOWNLOAD = "adaptive_intervention_budget_7"
# CONFIG_FOLDERS_TO_DOWNLOAD = "adaptive_intervention_budget_mechanism_15"
# CONFIG_FOLDERS_TO_DOWNLOAD = ["adaptive_intervention_budget_7", "gine_s0_s1_s2"]
# CONFIG_FOLDERS_TO_DOWNLOAD = "adaptive_intervention_budget_7,gine_s0_s1_s2"
# CONFIG_FOLDERS_TO_DOWNLOAD = "all"
CONFIG_FOLDERS_TO_DOWNLOAD = ["adaptive_intervention_budget_7",
                              "adaptive_intervention_budget_mechanism_15",
                              "gine_s0_s1_s2",
                              "entropy_decay_s0_s1_s2",
                              "no_entropy_decay_s0_s1_s2",
                              "gine_s0_s1_s2_include_neighbors",
                              "phase4_sparse_control_16",
                              "heuristic_vs_gate_s0_s1_s2",
                              "adaptive_intervention_budget_mechanism_15",]

# False only reads local cached histories. True downloads missing matching histories from W&B.
DOWNLOAD_MISSING_FROM_WANDB = False

# Useful when W&B has newer data than the local cache, especially for old scan_history fallback caches.
# This only has an effect when DOWNLOAD_MISSING_FROM_WANDB is True.
REFRESH_SCAN_HISTORY_FALLBACKS = False

# Heavier option: replace every selected local cache from W&B.
FORCE_REFRESH_CACHE = False

wm.configure_run_filter_from_config_folder(CONFIG_FOLDERS_TO_DOWNLOAD)
wm.EXCLUDE_RUN_NAME_REGEX = None
wm.RUN_STATES = None
wm.MAX_RUNS = None
wm.USE_LOCAL_CACHE_ONLY = not DOWNLOAD_MISSING_FROM_WANDB
wm.REFRESH_SCAN_HISTORY_FALLBACKS = bool(DOWNLOAD_MISSING_FROM_WANDB and REFRESH_SCAN_HISTORY_FALLBACKS)
wm.FORCE_REFRESH = bool(DOWNLOAD_MISSING_FROM_WANDB and FORCE_REFRESH_CACHE)
wm.ALLOW_SCAN_HISTORY_FALLBACK = True
wm.refresh_run_filters()

print("RUN_NAME_REGEX =", wm.RUN_NAME_REGEX)
print("USE_LOCAL_CACHE_ONLY =", wm.USE_LOCAL_CACHE_ONLY)
print("REFRESH_SCAN_HISTORY_FALLBACKS =", wm.REFRESH_SCAN_HISTORY_FALLBACKS)
print("FORCE_REFRESH =", wm.FORCE_REFRESH)



## Load Selected W&B Histories


In [ ]:
data = wm.load_wandb_data()

runs_df = data.runs_df
history_df = data.history_df
runs_df


## Entropy Decay


In [ ]:
entropy_decay_fig = wm.plot_entropy_decay_comparison()
entropy_decay_fig


## GINE Best Search 2


In [ ]:
gine_best2_fig = wm.plot_gine_best2_baseline_comparisons()
gine_best2_fig


## Heuristic vs Gate


In [ ]:
hvg_result = wm.plot_heuristic_vs_gate_survival()
hvg_result["fig"]


## Phase4 Sparse Control 16


In [ ]:
phase4_sparse_result = wm.plot_phase4_sparse_control_16_survival()
phase4_sparse_result["fig"]


## Adaptive Intervention Budget 7


In [ ]:
AIB_BASELINE_SOURCE = "gine_best2"  # options: "gine_best2", "hvg", "phase4_sparse"
aib_result = wm.plot_adaptive_intervention_budget_7_comparisons(baseline_source=AIB_BASELINE_SOURCE)
aib_result["fig"]


## Adaptive Intervention Budget Mechanism 15


In [ ]:
from IPython.display import display

aibm_result = wm.plot_adaptive_intervention_budget_mechanism_15()
for fig in [aibm_result["survival_fig"], aibm_result["diagnostic_fig"]]:
    if fig is not None:
        display(fig)


## GINE Include Neighbors


In [ ]:
gine_neighbor_result = wm.plot_gine_neighbors_vs_original()
gine_neighbor_result["fig"]
